#### 감정 분석 자연어 처리
1. data 폴더 안에 ratings_train.txt 파일을 로드
2. 데이터를 상위 500개 데이터만 추출
    - 데이터를 25000부터 3000번째의 데이터를 이용
3. 리뷰 데이터와 감정 데이터로 나눠준다.
4. 리뷰 데이터를 토큰화(Komoran함수 이용)
5. Word2Vec 학습
    - window -> 3
    - epochs -> 5
    - min_count -> 5
    - sg -> 1
    - seed -> 42
6. 벡터화(Word2Vec, 단위 벡터의 평균)
7. 분류 모델 (SVC, Logistic)
8. train, test을 이용하여 2개의 모델 중 성능이 높은 모델이 무엇인가?
9. 단위 벡터의 평균의 성능과 단위 벡터 + 중요도 평균의 성능의 차이를 확인

In [201]:
from gensim.models import Word2Vec
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from konlpy.tag import Komoran
from sklearn.svm import SVC
from sklearn import linear_model

In [202]:
data = pd.read_csv("../data/ratings_train.txt", sep="\t")
data.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [203]:
data.isna().sum()

id          0
document    5
label       0
dtype: int64

In [204]:
for i in data['document'][25000:30000]:
    if type(i) != str:
        print(type(i))

<class 'float'>


In [205]:
data['document'] = data['document'].fillna('')

In [206]:
data['document'].isna().sum()

np.int64(0)

In [207]:
# 리뷰 데이터와 감정 데이터로 분류
reviews = data['document'][:30000].values
emotions = data['label'][:30000].values

In [208]:
# 토큰화 함수 생성
# komoran 사용(konlpy 설치가 되어있는 경우)
# 설치가 되어있지 않은 경우에는 split()을 이용하여 토큰화
def bulid_tokenize():
    try:
        # 라이브러리 로드 -> 라이브러리가 존재하면 코드를 실행
        from konlpy.tag import Komoran
        komoran = Komoran()
        allow_pos= ['NNP', 'NNG', 'VV', 'VA', 'SL','MAG']
        def tokenize(text):
            tokens = []
            for word, pos in komoran.pos(text):
                if pos in allow_pos:
                    tokens.append(word)
            return tokens
        # tokenize 함수를 결과로 되돌려준다.
        return tokenize
    except Exception as e:
        print("Komoran 사용 불가 : ", e)
        return lambda x : x.split()

In [209]:
tokenize = bulid_tokenize()

In [210]:
tokens = [tokenize(rew) for rew in reviews]
tokens

[['더빙', '진짜', '짜증', '나', '목소리'],
 ['포스터', '초딩', '영화', '오버', '연기', '가볍'],
 [],
 ['교도소', '이야기', '솔직히', '재미', '없', '평점', '조정'],
 ['익살', '연기', '돋보이', '영화', '스파이더맨', '늙', '보이', '하', '커스틴 던스트', '너무나'],
 ['막', '걸음마', '떼', '초등학교', '학년', '아깝'],
 ['원작', '긴장감', '제대로', '살리'],
 ['반개',
  '아깝',
  '욕',
  '나오',
  '이응경',
  '길용우',
  '연기',
  '생활',
  '이',
  '정말',
  '발로',
  '납치',
  '감금',
  '반복',
  '반복',
  '이',
  '드라마',
  '가족',
  '없',
  '연기',
  '못하',
  '사람',
  '모이'],
 ['액션', '없', '재미', '있', '안', '영화'],
 ['왜', '평점', '낮', '꽤', '보', '헐리우드', '너무', '길들이'],
 [],
 ['볼',
  '때',
  '눈물',
  '나서',
  '죽',
  '향수',
  '자극',
  '!!',
  '허진호',
  '감성',
  '절제',
  '멜로',
  '달인',
  '이다'],
 ['울', '손들', '횡단보도', '건너', '때', '뛰쳐나오', '이범수', '연기', '드럽'],
 ['좋', '신문', '기사', '로만', '보다', '보', '자꾸', '잊어버리', '사람'],
 ['취향',
  '존중',
  '진짜',
  '극장',
  '보',
  '영화',
  '가장',
  '노',
  '재',
  '노',
  '감동',
  '스토리',
  '어거지',
  '감동',
  '어거지'],
 ['매번', '긴장'],
 ['참',
  '사람',
  '웃기',
  '바스코',
  '이기',
  '락스',
  '코',
  '까',
  '고',
  '바비',
  '이기',
  '아이돌',
  '

In [211]:
w2v = Word2Vec(
    sentences= tokens,
    window=5,
    min_count=2,
    epochs=100,
    seed = 42,
    sg=1
)
wv = w2v.wv

In [ ]:
def sent_embed_mean(tokens):
    vecs = []
    for word in tokens:
        if word in wv.index_to_key:
            vecs.append(wv[word])
    result = np.mean(vecs, axis=0) if vecs else np.zeros(wv.vector_size)
    return result

In [221]:
tfidf_vec = TfidfVectorizer(
    tokenizer=tokenize,
    lowercase=False
).fit(reviews)
tfidf_vec

c:\Users\abohv\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,False
,preprocessor,None
,tokenizer,<function bul...002C16130C4A0>
,analyzer,'word'
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


In [222]:
idf = dict(
    zip(
        # get_feature_names_out() -> Tfidf에서 사용된 단어들의 목록
        tfidf_vec.get_feature_names_out(),
        # idf_ : 중요도
        tfidf_vec.idf_
    )
)

In [223]:
# 단어 별 단위 벡터의 평균과 idf를 곱한다.
def sent_embed_tfidf(tokens):
    vecs = []
    weight = []
    for word in tokens:
        # tokens에 각각의 단어가 Word2Vec과 TF-IDF에 존재하ㅁㄴ다면
        if word in wv.key_to_index and word in idf:
            # vecs -> 단위벡터와 중요도를 곱한 값을 vecs 추가
            vecs.append(wv[word] * idf[word])
            # weight -> 중요도 데이터를 추가
            weight.append(idf[word])
    # vecs의 데이터가 존재하지 않는다면 -> tokens 안에 단어는 존재하지만 Word2Vec이나 TD-IDF에 단어가 존재하지 않을때
    if not vecs:
        # 희소 행렬 되돌려준다. 0행렬
        result = np.zeros(wv.vector_size)
    else:
        result = np.sum(vecs, axis=0) / (np.sum(weight) + 1e-9)
    return result

In [220]:
tfidf_vec = TfidfVectorizer(
    tokenizer=tokenize,
    lowercase=False
)

In [231]:
X_embed_mean = [sent_embed_mean(token) for token in tokens]
X_embed_mean

[array([ 4.45966333e-01, -2.06965417e-01, -2.03725137e-02,  9.09463763e-02,
        -1.13792196e-01, -2.93222874e-01,  6.50556684e-02,  4.07847688e-02,
        -2.28652328e-01,  4.24704731e-01, -2.34415263e-01,  1.24470711e-01,
         2.31077388e-01, -9.07217935e-02,  3.02760869e-01, -4.77830172e-02,
        -1.64164320e-01, -3.51802051e-01, -5.09542286e-01, -2.03945547e-01,
         3.67734805e-02,  5.38724288e-02,  7.03235641e-02,  2.45777279e-01,
         2.00679570e-01,  1.58796757e-01, -8.23448002e-01, -1.73201472e-01,
        -2.73322165e-01, -1.40286550e-01, -1.30006269e-01,  1.01769425e-01,
         1.63704917e-01, -6.51649833e-02,  6.60944283e-01, -4.05413061e-02,
         2.71189392e-01, -3.86636287e-01, -8.57906565e-02,  3.22001219e-01,
         9.83197093e-02, -5.50386321e-04,  4.47759718e-01, -1.43994838e-01,
         3.88814628e-01, -9.40144882e-02, -3.89208794e-02,  5.11592984e-01,
         6.92651570e-01, -2.17078060e-01,  3.15353453e-01, -3.57887477e-01,
        -3.5

In [232]:
X_embed_tfidf = [sent_embed_tfidf(token) for token in tokens]
X_embed_tfidf

[array([ 0.51426268, -0.22937688, -0.02161173,  0.08071013, -0.16004194,
        -0.33329455,  0.08631376,  0.06424462, -0.23053175,  0.44722115,
        -0.26764287,  0.1029834 ,  0.25597638, -0.11671522,  0.30887744,
        -0.10269484, -0.15166649, -0.41165694, -0.56925847, -0.16202992,
         0.05077856,  0.09716287,  0.05858495,  0.28257425,  0.21286533,
         0.16674221, -0.86587544, -0.21140831, -0.3044392 , -0.14607543,
        -0.1307924 ,  0.12789324,  0.19526902, -0.0671707 ,  0.67885435,
        -0.01712083,  0.28341848, -0.34679668, -0.08794   ,  0.37894415,
         0.06039338, -0.0052296 ,  0.47439243, -0.11656572,  0.39774406,
        -0.10479757, -0.05575864,  0.54775842,  0.80021468, -0.25887359,
         0.36990807, -0.35907887, -0.37141874, -0.25452462,  0.31255608,
        -0.24863009, -0.36510082,  0.19063721, -0.23237957,  0.02970046,
        -0.02832178, -0.04648435, -0.33313901,  0.17437581,  0.12295266,
        -0.46297736,  0.07654142, -0.0629048 ,  0.4

In [214]:
# 데이터의 개수가 적당히 많은 수준인 경우 Train, Test 데이터를 나눠주고
# Train 데이터를 이용하여 학습을 하고 Test 데이터를 이용해서 검증
svc = SVC(random_state=42)

def run_model(X, Y, test_size=0.2):
    # X는 독립 변수
    # Y는 종속 변수
    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=test_size, stratify=Y
    )
    # 모델에 학습
    svc.fit(X_train, Y_train)
    # 학습된 모델에 예측 값
    y_pred = svc.predict(X_test)
    print("정확도 : ", round(
        accuracy_score(y_pred, Y_test), 4)
    )
    print("분류 레포트 : ", classification_report(y_pred, Y_test))

In [ ]:
leg = linear_model.LogisticRegression(random_state=42)

def run_model2(X, Y, test_size=0.2):
    # X는 독립 변수
    # Y는 종속 변수
    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=test_size, stratify=Y
    )
    # 모델에 학습
    leg.fit(X_train, Y_train)
    # 학습된 모델에 예측 값
    y_pred = leg.predict(X_test)
    print("정확도 : ", round(
        accuracy_score(y_pred, Y_test), 4)
    )
    print("분류 레포트 : ", classification_report(y_pred, Y_test))

In [ ]:
run_model(X_embed_mean, emotions)

정확도 :  0.8027
분류 레포트 :                precision    recall  f1-score   support

           0       0.80      0.80      0.80      3011
           1       0.80      0.80      0.80      2989

    accuracy                           0.80      6000
   macro avg       0.80      0.80      0.80      6000
weighted avg       0.80      0.80      0.80      6000



In [ ]:
run_model2(X_embed_mean, emotions)

정확도 :  0.7807
분류 레포트 :                precision    recall  f1-score   support

           0       0.78      0.78      0.78      3019
           1       0.78      0.78      0.78      2981

    accuracy                           0.78      6000
   macro avg       0.78      0.78      0.78      6000
weighted avg       0.78      0.78      0.78      6000



In [225]:
run_model(X_embed_tfidf, emotions)

정확도 :  0.7838
분류 레포트 :                precision    recall  f1-score   support

           0       0.79      0.78      0.79      3026
           1       0.78      0.78      0.78      2974

    accuracy                           0.78      6000
   macro avg       0.78      0.78      0.78      6000
weighted avg       0.78      0.78      0.78      6000



In [226]:
run_model2(X_embed_tfidf, emotions)

정확도 :  0.7745
분류 레포트 :                precision    recall  f1-score   support

           0       0.77      0.78      0.77      2970
           1       0.78      0.77      0.78      3030

    accuracy                           0.77      6000
   macro avg       0.77      0.77      0.77      6000
weighted avg       0.77      0.77      0.77      6000



In [237]:
# sent_embed_tfidf() -> 출력은 머신러닝 모델에서 학습 데이터로 사용한 벡터 데이터
# 학습된 모델에 예측의 값을 반환 함수
# 세번째 매개변수(vec_type)를 생성 -> 기본값은 'mean'
# 'tfidf' 입력이 들어온다면 벡터화 작업은 w2v + ifidf 융합한 벡터화
def predict_sentence_list(sentences, model, vec_type = 'mean'):
    # sentneces : 문자들의 리스트
    # 문장들을 토큰화 -> 임베딩
    X_test = []
    for sent in sentences:
        # token() 함수를 호출하여 토큰화
        tokens = tokenize(sent)
        
        if vec_type == 'mean':
            vec = sent_embed_mean(tokens)
        elif vec_type == 'tfidf':
            vec = sent_embed_tfidf(tokens)
        X_test.append(vec)
        
    preds = model.predict(X_test)
    result = []
    for sent, pred in zip(sentences, preds):
        label = "긍정" if pred == 1 else "부정"
        result.append([sent, label])
    return result

In [238]:
# 모델 학습 -> 예측
X_test = data['document'].tail(10).values
run_model(X_embed_mean, emotions)

predict_sentence_list(X_test, svc, vec_type='mean')

정확도 :  0.7857
분류 레포트 :                precision    recall  f1-score   support

           0       0.79      0.78      0.79      3041
           1       0.78      0.79      0.78      2959

    accuracy                           0.79      6000
   macro avg       0.79      0.79      0.79      6000
weighted avg       0.79      0.79      0.79      6000



[['이걸 영화라고 찎었냐?', '부정'],
 ['http://blog.naver.com/oroblast/220215679580 나쁜 인상은 아니지만,오랫동안 기억에 남아....종종 떠올라서....조금은 사람을 피곤하게 만드는 영화. ^^',
  '긍정'],
 ['공포나 재난영화가 아니라 아예 대놓고 비급 크리쳐개그물임ㅋㅋ 음악 완전 흥겹다ㅋ 5점정도가 적당한 거 같은데 평점이 좀 높아서ㅋㅋ',
  '부정'],
 ['For Carl.칼 세이건으로 시작해서 칼 세이건으로 끝난다.', '부정'],
 ['디케이드 다음에 더블 다음에 오즈인데 더블은 조금밖에 안나오네요.', '부정'],
 ['인간이 문제지.. 소는 뭔죄인가..', '부정'],
 ['평점이 너무 낮아서...', '긍정'],
 ['이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?', '부정'],
 ['청춘 영화의 최고봉.방황과 우울했던 날들의 자화상', '긍정'],
 ['한국 영화 최초로 수간하는 내용이 담긴 영화', '긍정']]

In [239]:
run_model2(X_embed_tfidf, emotions)
predict_sentence_list(X_test, leg, vec_type='tfidf')

정확도 :  0.7692
분류 레포트 :                precision    recall  f1-score   support

           0       0.76      0.77      0.77      2954
           1       0.78      0.76      0.77      3046

    accuracy                           0.77      6000
   macro avg       0.77      0.77      0.77      6000
weighted avg       0.77      0.77      0.77      6000



[['이걸 영화라고 찎었냐?', '긍정'],
 ['http://blog.naver.com/oroblast/220215679580 나쁜 인상은 아니지만,오랫동안 기억에 남아....종종 떠올라서....조금은 사람을 피곤하게 만드는 영화. ^^',
  '긍정'],
 ['공포나 재난영화가 아니라 아예 대놓고 비급 크리쳐개그물임ㅋㅋ 음악 완전 흥겹다ㅋ 5점정도가 적당한 거 같은데 평점이 좀 높아서ㅋㅋ',
  '부정'],
 ['For Carl.칼 세이건으로 시작해서 칼 세이건으로 끝난다.', '부정'],
 ['디케이드 다음에 더블 다음에 오즈인데 더블은 조금밖에 안나오네요.', '부정'],
 ['인간이 문제지.. 소는 뭔죄인가..', '부정'],
 ['평점이 너무 낮아서...', '긍정'],
 ['이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?', '부정'],
 ['청춘 영화의 최고봉.방황과 우울했던 날들의 자화상', '긍정'],
 ['한국 영화 최초로 수간하는 내용이 담긴 영화', '긍정']]